# 📊 Notebook 1: Data Exploration & Understanding

## CWRU Bearing Fault Detection System

---

### Objectives
1. Load and understand the CWRU bearing dataset structure
2. Visualize raw vibration signals across all fault types
3. Perform time-domain and frequency-domain analysis
4. Understand class distributions and dataset characteristics
5. Extract statistical features for initial fault signature analysis

### Dataset: Case Western Reserve University (CWRU) Bearing Data
- **Bearing Type**: 6205-2RS JEM SKF deep groove ball bearing
- **Drive-End Accelerometer**: 12 kHz sampling rate
- **Motor Load**: 0 HP to 3 HP (1797–1720 RPM)
- **Fault Types**: Inner Race, Outer Race, Ball — at 0.007", 0.014", 0.021" diameters
- **10 Classes**: 1 Normal + 9 Fault conditions

## 1. Environment Setup

In [ ]:
# ============================================================
# Imports & Configuration
# ============================================================
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.io import loadmat
from scipy.signal import welch, spectrogram
from scipy.stats import kurtosis, skew
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Project modules
from src.data.data_loader import CWRUDataLoader
from src.data.preprocessing import bandpass_filter, normalize_signal, create_windows

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 100,
})

# Color palette for fault types
FAULT_COLORS = {
    'Normal': '#2ecc71',
    'Inner Race': '#e74c3c',
    'Outer Race': '#3498db',
    'Ball': '#f39c12'
}

print("✅ Environment ready!")
print(f"📁 Project root: {PROJECT_ROOT}")

## 2. Load the CWRU Dataset

In [ ]:
# ============================================================
# Load all .mat files using our data loader
# ============================================================
DATA_DIR = str(PROJECT_ROOT / 'data' / 'cwru')
loader = CWRUDataLoader(data_dir=DATA_DIR)

signals_dict, metadata_df = loader.load_all_data()

print(f"\n{'='*60}")
print(f"📦 Dataset Summary")
print(f"{'='*60}")
print(f"Total files loaded: {len(signals_dict)}")
print(f"Total classes: {metadata_df['class_id'].nunique()}")
print(f"\n📋 Metadata preview:")
metadata_df.head(10)

In [ ]:
# ============================================================
# Full metadata overview
# ============================================================
print("\n📊 Full Dataset Metadata:\n")
display_df = metadata_df[['filename', 'class_name', 'class_id', 
                          'sampling_rate', 'num_samples', 'duration_sec']].copy()
display_df['duration_sec'] = display_df['duration_sec'].round(2)
display_df = display_df.sort_values(['class_id', 'filename']).reset_index(drop=True)
display_df

## 3. Class Distribution Analysis

In [ ]:
# ============================================================
# Class distribution & signal length analysis
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a. Files per class
class_counts = metadata_df['class_name'].value_counts().sort_index()
colors = []
for name in class_counts.index:
    if 'Normal' in name:
        colors.append(FAULT_COLORS['Normal'])
    elif 'Inner' in name:
        colors.append(FAULT_COLORS['Inner Race'])
    elif 'Outer' in name:
        colors.append(FAULT_COLORS['Outer Race'])
    else:
        colors.append(FAULT_COLORS['Ball'])

axes[0].barh(class_counts.index, class_counts.values, color=colors, 
             edgecolor='black', linewidth=0.5, alpha=0.8)
axes[0].set_xlabel('Number of Files')
axes[0].set_title('Files per Fault Class', fontweight='bold')
for i, v in enumerate(class_counts.values):
    axes[0].text(v + 0.1, i, str(v), va='center', fontweight='bold')

# 3b. Signal duration per class
metadata_df.boxplot(column='duration_sec', by='class_name', ax=axes[1],
                    vert=True, patch_artist=True)
axes[1].set_title('Signal Duration per Class', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Duration (seconds)')
axes[1].tick_params(axis='x', rotation=45)
plt.suptitle('')  # Remove auto title

# 3c. Total samples per class
total_samples = metadata_df.groupby('class_name')['num_samples'].sum().sort_index()
axes[2].bar(range(len(total_samples)), total_samples.values / 1e6, 
            color=colors, edgecolor='black', linewidth=0.5, alpha=0.8)
axes[2].set_xticks(range(len(total_samples)))
axes[2].set_xticklabels(total_samples.index, rotation=45, ha='right')
axes[2].set_ylabel('Total Samples (Millions)')
axes[2].set_title('Total Samples per Class', fontweight='bold')

plt.tight_layout()
plt.show()

# Summary statistics
print("\n📈 Signal Length Statistics:")
print(f"  Min duration: {metadata_df['duration_sec'].min():.2f} s")
print(f"  Max duration: {metadata_df['duration_sec'].max():.2f} s")
print(f"  Mean duration: {metadata_df['duration_sec'].mean():.2f} s")
print(f"  Total data: {metadata_df['num_samples'].sum():,} samples")

## 4. Raw Signal Visualization

Let's visualize one representative signal from each of the 10 classes to understand the visual characteristics of different fault types.

In [ ]:
# ============================================================
# Plot one signal per class (raw, first 0.1 seconds)
# ============================================================
CLASS_LABELS = loader.CLASS_LABELS
FS = 12000  # Sampling frequency after downsampling

fig, axes = plt.subplots(5, 2, figsize=(16, 20), sharex=True)
axes = axes.flatten()

# Get one representative file per class
representative_files = {}
for fname, class_id in loader.FILE_MAPPING.items():
    if class_id not in representative_files and fname in signals_dict:
        representative_files[class_id] = fname

for class_id in sorted(representative_files.keys()):
    fname = representative_files[class_id]
    signal = signals_dict[fname]
    
    # Show first 0.1 seconds (1200 samples at 12 kHz)
    n_show = min(1200, len(signal))
    time_axis = np.arange(n_show) / FS * 1000  # in milliseconds
    
    ax = axes[class_id]
    
    # Determine color
    if class_id == 0:
        color = FAULT_COLORS['Normal']
    elif class_id <= 3:
        color = FAULT_COLORS['Inner Race']
    elif class_id <= 6:
        color = FAULT_COLORS['Outer Race']
    else:
        color = FAULT_COLORS['Ball']
    
    ax.plot(time_axis, signal[:n_show], linewidth=0.6, color=color)
    ax.set_title(f'Class {class_id}: {CLASS_LABELS[class_id]} ({fname})', 
                 fontweight='bold', fontsize=11)
    ax.set_ylabel('Amplitude (g)')
    ax.grid(True, alpha=0.3)
    
    # Add RMS and Peak annotations
    rms = np.sqrt(np.mean(signal[:n_show]**2))
    peak = np.max(np.abs(signal[:n_show]))
    ax.text(0.98, 0.95, f'RMS={rms:.3f}  Peak={peak:.3f}', 
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[-1].set_xlabel('Time (ms)')
axes[-2].set_xlabel('Time (ms)')

plt.suptitle('Raw Vibration Signals — One Per Fault Class (First 100 ms)', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Fault Type Comparison

Compare signals from the same fault type at different severity levels (0.007", 0.014", 0.021").

In [ ]:
# ============================================================
# Severity comparison: Inner Race faults
# ============================================================
fault_groups = {
    'Inner Race': [0, 1, 2, 3],   # Normal, IR007, IR014, IR021
    'Outer Race': [0, 4, 5, 6],   # Normal, OR007, OR014, OR021
    'Ball':       [0, 7, 8, 9],   # Normal, B007, B014, B021
}

for group_name, class_ids in fault_groups.items():
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    severities = ['Normal', '0.007"', '0.014"', '0.021"']
    
    for i, (class_id, severity) in enumerate(zip(class_ids, severities)):
        fname = representative_files.get(class_id)
        if fname is None:
            continue
            
        signal = signals_dict[fname]
        n_show = 2048
        time_axis = np.arange(n_show) / FS * 1000
        
        color = FAULT_COLORS.get(group_name, '#666666') if class_id > 0 else FAULT_COLORS['Normal']
        alpha = 0.4 + 0.2 * i  # Increasing opacity with severity
        
        axes[i].plot(time_axis, signal[:n_show], linewidth=0.7, color=color)
        axes[i].set_title(f'{CLASS_LABELS[class_id]} (Severity: {severity})', fontweight='bold')
        axes[i].set_ylabel('Amplitude (g)')
        axes[i].set_xlabel('Time (ms)')
        axes[i].grid(True, alpha=0.3)
        
        # Stats
        seg = signal[:n_show]
        rms = np.sqrt(np.mean(seg**2))
        kurt = kurtosis(seg)
        axes[i].text(0.98, 0.95, f'RMS={rms:.4f}\nKurtosis={kurt:.2f}', 
                     transform=axes[i].transAxes, ha='right', va='top',
                     fontsize=9, bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    
    plt.suptitle(f'{group_name} Fault — Severity Comparison', 
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## 6. Frequency Domain Analysis

The frequency content is critical for bearing fault detection. Each fault type creates characteristic frequency signatures related to the bearing geometry.

### Bearing Characteristic Frequencies (CWRU 6205-2RS @ 1797 RPM)
| Frequency | Formula | Value |
|-----------|---------|-------|
| BPFI (Inner Race) | $\frac{n}{2} f_r (1 + \frac{d}{D} \cos\alpha)$ | ~162 Hz |
| BPFO (Outer Race) | $\frac{n}{2} f_r (1 - \frac{d}{D} \cos\alpha)$ | ~107 Hz |
| BSF (Ball) | $\frac{D}{2d} f_r (1 - (\frac{d}{D} \cos\alpha)^2)$ | ~140 Hz |
| FTF (Cage) | $\frac{1}{2} f_r (1 - \frac{d}{D} \cos\alpha)$ | ~12 Hz |

In [ ]:
# ============================================================
# Bearing characteristic frequencies
# ============================================================
SHAFT_RPM = 1797
SHAFT_HZ = SHAFT_RPM / 60
N_BALLS = 9
D_BALL = 0.312  # inches
D_PITCH = 1.537  # inches

BPFI = (SHAFT_HZ / 2) * N_BALLS * (1 + D_BALL / D_PITCH)
BPFO = (SHAFT_HZ / 2) * N_BALLS * (1 - D_BALL / D_PITCH)
BSF  = (SHAFT_HZ / 2) * (D_PITCH / D_BALL) * (1 - (D_BALL / D_PITCH)**2)
FTF  = (SHAFT_HZ / 2) * (1 - D_BALL / D_PITCH)

print("⚙️ Bearing Characteristic Frequencies:")
print(f"  Shaft Speed:   {SHAFT_HZ:.1f} Hz ({SHAFT_RPM} RPM)")
print(f"  BPFI (Inner):  {BPFI:.1f} Hz")
print(f"  BPFO (Outer):  {BPFO:.1f} Hz")
print(f"  BSF (Ball):    {BSF:.1f} Hz")
print(f"  FTF (Cage):    {FTF:.1f} Hz")

In [ ]:
# ============================================================
# Power Spectral Density — All classes
# ============================================================
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()

for class_id in sorted(representative_files.keys()):
    fname = representative_files[class_id]
    signal = signals_dict[fname]
    
    # Compute PSD
    freqs, psd = welch(signal, fs=FS, nperseg=2048)
    
    ax = axes[class_id]
    
    # Determine color
    if class_id == 0:
        color = FAULT_COLORS['Normal']
    elif class_id <= 3:
        color = FAULT_COLORS['Inner Race']
    elif class_id <= 6:
        color = FAULT_COLORS['Outer Race']
    else:
        color = FAULT_COLORS['Ball']
    
    ax.semilogy(freqs, psd, color=color, linewidth=0.8)
    ax.set_title(f'{CLASS_LABELS[class_id]}', fontweight='bold')
    ax.set_ylabel('PSD (g²/Hz)')
    ax.set_xlim([0, 2000])
    ax.grid(True, alpha=0.3)
    
    # Mark characteristic frequencies
    ax.axvline(x=BPFI, color='red', linestyle='--', alpha=0.5, linewidth=1, label='BPFI')
    ax.axvline(x=BPFO, color='blue', linestyle='--', alpha=0.5, linewidth=1, label='BPFO')
    ax.axvline(x=BSF, color='green', linestyle='--', alpha=0.5, linewidth=1, label='BSF')
    
    if class_id == 0:
        ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Frequency (Hz)')
axes[-2].set_xlabel('Frequency (Hz)')

plt.suptitle('Power Spectral Density — All Fault Classes (0–2000 Hz)', 
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Spectrogram visualization — one per fault type
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

example_classes = [0, 1, 4, 7]  # Normal, Inner, Outer, Ball
titles = ['Normal', 'Inner Race 0.007"', 'Outer Race 0.007"', 'Ball 0.007"']

for i, (class_id, title) in enumerate(zip(example_classes, titles)):
    fname = representative_files[class_id]
    signal = signals_dict[fname]
    
    # Compute spectrogram
    f, t, Sxx = spectrogram(signal[:24000], fs=FS, nperseg=512, noverlap=448)
    
    ax = axes[i]
    im = ax.pcolormesh(t, f, 10 * np.log10(Sxx + 1e-12), 
                       shading='gouraud', cmap='inferno')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim([0, 3000])
    plt.colorbar(im, ax=ax, label='Power (dB)')

plt.suptitle('Spectrograms — Fault Type Comparison', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 7. Statistical Feature Analysis

Extract time-domain statistical features to understand separability between classes.

In [ ]:
# ============================================================
# Extract features from all signals
# ============================================================
feature_records = []

for fname, signal in signals_dict.items():
    class_id = loader.FILE_MAPPING[fname]
    class_name = CLASS_LABELS[class_id]
    
    # Extract windows and compute features per window
    windows, _ = create_windows(signal, window_size=2048, overlap=0.5)
    
    for win in windows[:50]:  # Sample first 50 windows per file
        record = {
            'filename': fname,
            'class_id': class_id,
            'class_name': class_name,
            'fault_type': 'Normal' if class_id == 0 else class_name.rsplit('_', 1)[0].replace('_', ' '),
            'rms': np.sqrt(np.mean(win**2)),
            'peak': np.max(np.abs(win)),
            'peak_to_peak': np.max(win) - np.min(win),
            'crest_factor': np.max(np.abs(win)) / (np.sqrt(np.mean(win**2)) + 1e-8),
            'kurtosis': kurtosis(win),
            'skewness': skew(win),
            'std': np.std(win),
            'variance': np.var(win),
            'energy': np.sum(win**2),
            'zero_crossing_rate': np.sum(np.abs(np.diff(np.sign(win)))) / (2 * len(win)),
        }
        feature_records.append(record)

features_df = pd.DataFrame(feature_records)
print(f"📊 Extracted features from {len(features_df):,} windows")
print(f"   Features per window: {len(features_df.columns) - 4}")
features_df.head()

In [ ]:
# ============================================================
# Feature distributions by fault type
# ============================================================
feature_names = ['rms', 'peak', 'crest_factor', 'kurtosis', 'skewness', 'energy']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(feature_names):
    ax = axes[i]
    
    for fault_type, color in FAULT_COLORS.items():
        mask = features_df['fault_type'] == fault_type
        if mask.sum() > 0:
            data = features_df.loc[mask, feat].values
            ax.hist(data, bins=50, alpha=0.5, color=color, label=fault_type, density=True)
    
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Distributions by Fault Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Feature boxplots by class
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(feature_names):
    ax = axes[i]
    features_df.boxplot(column=feat, by='class_name', ax=ax,
                        patch_artist=True, vert=True, showfliers=False)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Feature Distributions by Class (Boxplots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Feature correlation heatmap
# ============================================================
numeric_features = features_df[feature_names + ['zero_crossing_rate', 'peak_to_peak', 'std']]

fig, ax = plt.subplots(figsize=(10, 8))
corr = numeric_features.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Scatter Plots — Separability Analysis

In [ ]:
# ============================================================
# 2D scatter plots of key feature pairs
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

scatter_pairs = [('rms', 'kurtosis'), ('peak', 'crest_factor'), ('energy', 'skewness')]

for ax, (feat_x, feat_y) in zip(axes, scatter_pairs):
    for fault_type, color in FAULT_COLORS.items():
        mask = features_df['fault_type'] == fault_type
        ax.scatter(features_df.loc[mask, feat_x], 
                   features_df.loc[mask, feat_y],
                   c=color, label=fault_type, alpha=0.4, s=10, edgecolors='none')
    
    ax.set_xlabel(feat_x.replace('_', ' ').title())
    ax.set_ylabel(feat_y.replace('_', ' ').title())
    ax.set_title(f'{feat_x} vs {feat_y}', fontweight='bold')
    ax.legend(fontsize=9, markerscale=3)
    ax.grid(True, alpha=0.3)

plt.suptitle('Feature Space — Class Separability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Signal Stationarity Check

Verify whether signals are stationary — important for our time-based split strategy.

In [ ]:
# ============================================================
# Rolling RMS to check stationarity
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

check_classes = [0, 1, 4, 7]  # Normal, Inner, Outer, Ball

for i, class_id in enumerate(check_classes):
    fname = representative_files[class_id]
    signal = signals_dict[fname]
    
    # Compute rolling RMS with window of 1024 samples
    window = 1024
    rolling_rms = np.array([
        np.sqrt(np.mean(signal[j:j+window]**2))
        for j in range(0, len(signal) - window, window // 4)
    ])
    
    time_axis = np.arange(len(rolling_rms)) * (window // 4) / FS
    
    ax = axes[i]
    ax.plot(time_axis, rolling_rms, linewidth=1, 
            color=list(FAULT_COLORS.values())[i])
    ax.axhline(y=np.mean(rolling_rms), color='black', linestyle='--', 
               alpha=0.5, label=f'Mean={np.mean(rolling_rms):.4f}')
    ax.fill_between(time_axis, 
                    np.mean(rolling_rms) - np.std(rolling_rms),
                    np.mean(rolling_rms) + np.std(rolling_rms),
                    alpha=0.2, color='gray', label=f'±1σ')
    ax.set_title(f'{CLASS_LABELS[class_id]} — Rolling RMS', fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('RMS')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Signal Stationarity Analysis (Rolling RMS)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Key Findings & Takeaways

### Dataset Characteristics
- **40 .mat files** across 10 classes (4 files per class)
- **Balanced dataset**: Each class has the same number of recording files
- Signals are at **12 kHz** sampling rate (48 kHz downsampled for files ≥ 169)
- Signal durations: ~10 seconds each → thousands of 2048-sample windows

### Signal Observations
- **Normal** signals have low amplitude, nearly Gaussian noise characteristics
- **Faulty** signals show periodic impulses at characteristic frequencies
- Severity progression is visible: amplitude increases with fault size
- **Kurtosis** is particularly discriminative between Normal and Faulty conditions

### Frequency Domain
- BPFI, BPFO, BSF harmonics are visible in respective fault types 
- Spectrograms show consistent frequency content → signals are approximately stationary

### Feature Separability
- RMS, Kurtosis, Crest Factor show good separability between fault types
- Some overlap between Ball faults and Outer Race faults → deep learning needed

### Implications for Model Design
- 1D-CNN can learn temporal patterns directly from raw signals
- Time-based split ensures no data leakage from the same recording
- Window size of 2048 samples (0.17s) captures multiple fault impulse cycles

In [ ]:
# ============================================================
# Summary statistics table
# ============================================================
summary = features_df.groupby('class_name')[feature_names].agg(['mean', 'std']).round(4)
print("\n📋 Feature Summary Statistics by Class:\n")
summary